<a href="https://colab.research.google.com/github/Chuuya1124/APM1201/blob/main/Lab_FA2_Awit%2C_JT_%26_Deloyola%2C_JR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# Assessment: Data Wrangling in R (tidyverse)
# Dataset: dplyr::starwars
# ============================================================================


# SETUP (REQUIRED)


library(tidyverse)
data("starwars", package = "dplyr")
# SETUP (REQUIRED)
library(tidyverse)
suppressMessages(library(tidyverse))

# Q1 — Dataset Inspection and Missing Values


cat("\n========================================\n")
cat("Q1 — Dataset Inspection and Missing Values\n")
cat("========================================\n\n")

# 1. Display the structure of the dataset
cat("1. Structure of the dataset:\n")
glimpse(starwars)

# 2. Identify number of observations and variables
cat("\n2. Dataset dimensions:\n")
n_obs <- nrow(starwars)
n_vars <- ncol(starwars)
cat("   Number of observations:", n_obs, "\n")
cat("   Number of variables:", n_vars, "\n")

# 3. Compute the number of missing values
cat("\n3. Missing values in height, mass, and homeworld:\n")
missing_summary <- starwars %>%
  summarise(
    missing_height = sum(is.na(height)),
    missing_mass = sum(is.na(mass)),
    missing_homeworld = sum(is.na(homeworld))
  )
print(missing_summary)


# Q2 — Create a Wide Summary Table (pivot_wider)


cat("\n\n========================================\n")
cat("Q2 — Create a Wide Summary Table\n")
cat("========================================\n\n")

wide_table <- starwars %>%
  # 1. Filter for non-missing species
  filter(!is.na(species)) %>%
  # 2. Group by species and gender
  # 3. Compute mean height
  group_by(species, gender) %>%
  summarise(mean_height = mean(height, na.rm = TRUE), .groups = "drop") %>%
  # 4. Pivot wider
  pivot_wider(
    names_from = gender,
    values_from = mean_height
  )

print(wide_table)


# Q3 — Convert Wide Table Back to Long Format (pivot_longer)


cat("\n\n========================================\n")
cat("Q3 — Convert Wide Table Back to Long Format\n")
cat("========================================\n\n")

long_table <- wide_table %>%
  pivot_longer(
    cols = -species,           # All columns except species
    names_to = "gender",       # Column names go to 'gender'
    values_to = "mean_height"  # Values go to 'mean_height'
  ) %>%
  filter(!is.na(mean_height))  # Remove rows with missing mean_height

print(long_table)


# Q4 — Create New Variables and Handle Missing Data


cat("\n\n========================================\n")
cat("Q4 — Create New Variables and Handle Missing Data\n")
cat("========================================\n\n")

starwars_modified <- starwars %>%
  mutate(
    # 1. Create BMI variable: BMI = mass / (height/100)²
    # height is in cm, so divide by 100 to convert to meters
    bmi = mass / ((height / 100)^2),

    # 2. Categorize height
    height_category = case_when(
      is.na(height) ~ NA_character_,
      height < 170 ~ "short",              # short: <170 cm
      height >= 170 & height <= 189 ~ "average",  # average: 170-189 cm
      height >= 190 ~ "tall"               # tall: ≥190 cm
    ),

    # 3. Replace missing homeworld with "Unknown"
    homeworld = replace_na(homeworld, "Unknown")
  )

cat("Modified dataset structure:\n")
glimpse(starwars_modified)


# Q5 — Unnesting and Interpretation (6 points)

cat("\n\n========================================\n")
cat("Q5 — Unnesting and Interpretation\n")
cat("========================================\n\n")

# 1-4. Select name and films, unnest, count, and display top 8
film_appearances <- starwars %>%
  select(name, films) %>%           # 1. Select relevant columns
  unnest(films) %>%                 # 2. Convert list-column to long format
  group_by(name) %>%                # 3. Group by character name
  summarise(n_films = n()) %>%      # 3. Count film appearances
  arrange(desc(n_films)) %>%        # 4. Sort by count (descending)
  slice_head(n = 8)                 # 4. Take top 8

cat("Top 8 characters by number of film appearances:\n")
print(film_appearances)





Q1 — Dataset Inspection and Missing Values

1. Structure of the dataset:
Rows: 87
Columns: 14
$ name       <chr> "Luke Skywalker", "C-3PO", "R2-D2", "Darth Vader", "Leia Or…
$ height     <int> 172, 167, 96, 202, 150, 178, 165, 97, 183, 182, 188, 180, 2…
$ mass       <dbl> 77.0, 75.0, 32.0, 136.0, 49.0, 120.0, 75.0, 32.0, 84.0, 77.…
$ hair_color <chr> "blond", NA, NA, "none", "brown", "brown, grey", "brown", N…
$ skin_color <chr> "fair", "gold", "white, blue", "white", "light", "light", "…
$ eye_color  <chr> "blue", "yellow", "red", "yellow", "brown", "blue", "blue",…
$ birth_year <dbl> 19.0, 112.0, 33.0, 41.9, 19.0, 52.0, 47.0, NA, 24.0, 57.0, …
$ sex        <chr> "male", "none", "none", "male", "female", "male", "female",…
$ gender     <chr> "masculine", "masculine", "masculine", "masculine", "femini…
$ homeworld  <chr> "Tatooine", "Tatooine", "Naboo", "Tatooine", "Alderaan", "T…
$ species    <chr> "Human", "Droid", "Droid", "Human", "Human", "Human", "Huma…
$ films      <list> <"A N

# **Why must this data b converted to long format?**

The data must be converted to long format because the films column is a "list-column", meaning it holds multiple movies inside a single cell. By unnesting it, we give every movie appearance its own row, which allows the computer to easily count them up and see which character appears most often."
